In [1]:
import sys
sys.path.append("..")

from src.data import load_data

df = load_data()
df.groupby("State")["Profit"].mean().round(0)

State
California    103905.0
Florida       118774.0
New York      113756.0
Name: Profit, dtype: float64

In [2]:
from src.data import split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

FEATURES = ["R&D Spend", "Administration", "Marketing Spend", "State"]
X_train, X_test, y_train, y_test = split(df, FEATURES)

prep = ColumnTransformer([
    ("state", OneHotEncoder(drop="first", handle_unknown="ignore"), ["State"]),
], remainder="passthrough")

lr_state = Pipeline([("prep", prep), ("model", LinearRegression())])
lr_state.fit(X_train, y_train)

y_pred = lr_state.predict(X_test)
cv = -cross_val_score(lr_state, X_train, y_train, cv=5,
                      scoring="neg_root_mean_squared_error").mean()

print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred):,.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.3f}")
print(f"CV RMSE:   {cv:,.0f}")

Test RMSE: 9,056
Test R²:   0.899
CV RMSE:   10,362
